<a href="https://colab.research.google.com/github/SauravSG/Deep-Learning/blob/main/W7_Fine_tuning_With_HF_Trainer_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **How we would train a sequence(sentence/sentences) classifier on one batch**

Earlier we explored how to use tokenisers and pretrained model to make predictions or generate texts. But what if we want to fine-tune a pretrained model to solve a specific task?

Let's see

In [11]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification # -> model by huggingface specifically for classification of text

In [12]:
# Assign the model
model_name = 'bert-base-uncased'

 # Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
sequences = [
    "I've been waiting for this Marvel movie to release.",
    "This movie is so amazing!"
]

tokens = tokenizer(
    sequences,
    padding = True,
    truncation = True,
    return_tensors = 'pt'
)

print(tokens)

In [ ]:
text1 = ['Dr Doom is releasing soon', 'I have not watched spiderman new movie yet']
test_token = tokenizer(text1, padding = True , truncation = True, return_tensors='pt')
print(test_token)

In [ ]:
tokens['labels'] = torch.tensor([1,1])

optimizer = AdamW(model.parameters())
print(optimizer)


In [ ]:
a = torch.tensor([1,1])
print(a)

In [ ]:
b = torch.randint(2, 4, (2, 4))
print(b)

In [ ]:
output_check = model(**tokens)
output_check

In [ ]:
loss = model(**tokens).loss
loss

In [ ]:
loss.backward()

In [ ]:
type(tokens)

In [ ]:
dictofsmth = {'arrayoffruits': 'apple, banana, dragonfruit',
              'prices': [2, 5, 6]}
print(dictofsmth)

In [ ]:
smth = {**dictofsmth}
print(smth)
smth2 = {*dictofsmth['arrayoffruits']}
print(smth2)

# ****

We will be using MRPC (Microsoft Research Paraphrase Corpus) dataset which consist of 5801 pairs of sentences, with label indicating if they are paraphrases or not (i.e. if both sentences has the same meaning)

We will be using dataset from GLUE.

- GLUE = General Language Understanding Evaluation — a collection of NLP benchmark tasks
- `path` = the Hub repo (`nyu-mll/glue`), `name` = which GLUE subset/config to load (e.g. `mrpc`, `sst2`, `cola`)
- **MRPC** subset = Microsoft Research Paraphrase Corpus → sentence-pair classification
  - Fields: `sentence1`, `sentence2`, `label`, `idx`
  - `label`: 1 = paraphrase (same meaning), 0 = not a paraphrase
  - `row`: Just a row identifier.

In [13]:
# import the load_dataset library
from datasets import load_dataset

In [14]:
raw_dataset = load_dataset(path='nyu-mll/glue', name='mrpc')
print(raw_dataset)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


In [3]:
raw_dataset['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

Let's check if the first row's two sentences of the train dataset has the same meaning or not:

In [ ]:
raw_train_dataset = raw_dataset['train'][0]
print(f'Checking if the two sentences from first row has the same meaning or not: \n')
print(raw_train_dataset)

`.features` shows the schema i.e. the column names, data types, how integer codes map to human-readable class names.

In [ ]:
raw_dataset['train'].features

The below will give **ClassLabel**

In [ ]:
raw_dataset['train'].features['label']

In [ ]:

raw_dataset['train'].features['label'].int2str(1)

In [ ]:
print(raw_dataset['train'][0]['label'])

Exploring Dataset Features & Labels

- `.features` shows the **schema** (column names + types) — not the actual data
  - e.g. `label: ClassLabel(names=['not_equivalent', 'equivalent'])` is the *legend*, not per-row values
- Actual row data is accessed via indexing: `raw_dataset['train'][0]['label']` → returns the raw int (e.g. `1`)
- `ClassLabel` int ↔ string lookups:
  - `int2str(1)` → `'equivalent'` (int → string)
  - `str2int('equivalent')` → `1` (string → int; passing an int here throws an error)
- `raw_dataset['train']['label']` returns a `Column([...])` object (Arrow-backed, list-like) — printed output may look truncated (e.g. `[1, 0, 1, 0, 1]`) but `len()` confirms it holds all rows; it's just a display convention, like numpy/pandas truncated repr
- Distinction: this is **categorical int-encoding** (`ClassLabel`), not boolean — boolean is strictly `True`/`False`

In [ ]:
print(f"To see what features/columns the train section of raw dataset has: \n{raw_dataset['train'].features}\n")

print(f"This would ideally give total 3668 number of labels but by default it truncate to 5 entries: \n {raw_dataset['train']['label']} \n")

print(f"Print labels of the first raw of the train section of raw dataset: \n {raw_dataset['train'][0]['label']} \n")

print(f"Print the string value or the Class Label: \n {raw_dataset['train'].features['label']} \n")

print(f"To see what the boolean value (0 and 1) means: \n {raw_dataset['train'].features['label'].int2str(0)} and {raw_dataset['train'].features['label'].int2str(1)} \n")

## **Tokenize Sentence Pair**

- Before this we will first see how we tokenize one pair manually

In [ ]:
# import tokenizer
from transformers import AutoTokenizer

# define and get the model
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Get the sentences
sentence1 = raw_dataset['train'][0]['sentence1']
sentence2 = raw_dataset['train'][1]["sentence2"]

# tokenize the sentences
tokenized_sentences = tokenizer(sentence1, sentence2)

# print the tokenized sentences
print(f"Sentences after tokenized: \n {tokenized_sentences}")

In [ ]:
tokenizer.convert_ids_to_tokens(tokenized_sentences['input_ids'])

In [ ]:
print(len(raw_dataset['train'][0]['sentence1']))
print(len(raw_dataset['train'][1]['sentence1']))

In [ ]:
tokenizer([raw_dataset['train'][0]['sentence1'], raw_dataset['train'][1]['sentence1']], padding = True)

In [ ]:
tokenized_1 = tokenizer(sentence1)
print(len(tokenized_1['attention_mask']))

In [ ]:
tokenized_2 = tokenizer(sentence2)
print(len(tokenized_2['attention_mask']))

In [18]:
def tokenize_function(example: dict):

  tokenized_output = tokenizer(example['sentence1'], example['sentence2'], truncation = True)

  return tokenized_output

In [19]:
result = tokenize_function(raw_dataset['train'][:2])
print(f"The first two rows of train dataset are: \n{raw_dataset['train'][:2]} \n and the result after tokenizing those two rows we get: \n{result} \n")

The first two rows of train dataset are: 
{'sentence1': ['Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion ."], 'sentence2': ['Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', "Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 ."], 'label': [1, 0], 'idx': [0, 1]} 
 and the result after tokenizing those two rows we get: 
{'input_ids': [[101, 2572, 3217, 5831, 5496, 2010, 2567, 1010, 3183, 2002, 2170, 1000, 1996, 7409, 1000, 1010, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102, 7727, 2000, 2032, 2004, 2069, 1000, 1996, 7409, 1000, 1010, 2572, 3217, 5831, 5496, 2010, 2567, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102], [101, 9805, 3540, 11514, 2050, 3079, 11282, 2243, 1005, 1055, 2077, 4855, 1996, 4677, 2000, 3647, 4

In [ ]:
raw_dataset['train'][:2]

In [20]:
tokenized_datasets = raw_dataset.map(tokenize_function, batched = True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets

In [ ]:
print(tokenized_datasets['train'][0])

## **Tokenizing Sentence Pairs + `tokenize_function` + `.map()`**

- **Tokenizing a pair**: `tokenizer(sentence1, sentence2)` → produces `[CLS] sentence1 [SEP] sentence2 [SEP]`
- **`token_type_ids`**: 0 for segment A (sentence1 + its closing [SEP]), 1 for segment B (sentence2 + final [SEP])
- **`attention_mask`**: 1 = real token, 0 = padding only appears when tokenizing a **batch** with `padding=True`; single examples have no padding needed
- **`tokenize_function(example)`**: takes a batch dict (`example['sentence1']` is a list of strings), returns tokenizer output for the pair — no padding here, since global padding wastes compute; padding happens later per-batch via `DataCollatorWithPadding`
- **`.map(tokenize_function, batched=True)`**: applies function across all splits, adds new columns (`input_ids`, `token_type_ids`, `attention_mask`) without dropping existing ones or changing row count
- Raw text columns (`sentence1`, `sentence2`, `idx`) are kept for inspection but removed later via `remove_columns` right before training, since `Trainer` needs only tensor-compatible columns

- `batched = True`: tells `.map()` to pass the function a chunk of rows at once (as lists), instead of one row at a time. Faster, since the tokenizer processed many sentences in one call rather looping.

- `attention_mask`: a 1/0 array marking which tokens are real(1) vs padding(0), so the model knows what to actually pay attention to.

- `truncation`: cuts off sequences longer than the model's max length (e.g BERT's 512 tokens), so overly long inputs son't break the model.

- `padding` - fills shorter sequences with `[PAD]` tokens so all sequences in a batch have the same length, making a valid tensor shape. This also helps the model to know which tokens are not real hence don't pay attention to those tokens (`[PAD]`).

## **DataCollatorWithPadding**

- It dynamically adds the padding length based on the maximum sentence input amongst the batch. This results in no wastage of extra tokens i.e. less GPU memory wasted on [PAD] tokens.

- We will see in the further examples how this works in the cells.

- For this dynamic padding, it mainly needs list of dictionary where the list contains `input_ids`, `token_type_ids`, `attention_mask` as these are the only pieces the model actually reads to make predictions.

  - `input_ids` -> the words themselves (in numbers).
  - `token_type_ids` -> which sentence each word belongs to.
  - `attention_mask` -> which words are real vs just padding filler

  The rest of the stuff like `sentence1`/ `sentence2`, `label`, `idx` we keep around for debugging or reference. The model also has no use of these in order to train and infer.

- So the Collator only asks for what it needs to pad and hand off to the model.

In [8]:
from transformers import DataCollatorWithPadding

In [15]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [ ]:
type(data_collator)

In [16]:
print(tokenized_datasets['train'][0])

NameError: name 'tokenized_datasets' is not defined

In [ ]:
samples = [tokenized_datasets['train'][i] for i in range(3)]
print(samples)

In [ ]:
refined_samples = {'input_ids': samples[0]['input_ids'],
 'token_type_ids': samples[0]['token_type_ids'],
 'attention_mask': samples[0]['attention_mask']}

In [ ]:
refined_samples['input_ids']

In [ ]:
new_dict_samples = [
    {'input_ids': samples[i]['input_ids'],
     'token_type_ids': samples[i]['token_type_ids'],
     'attention_mask': samples[i]['attention_mask']}

    for i in range(3)
]

In [ ]:
new_dict_samples[0]['input_ids']

In [ ]:
data_collator(new_dict_samples)

In [ ]:
# quick comparison between manual and dynamic padding

sentence1_batch = [raw_dataset['train'][i]['sentence1'] for i in range(3)]
sentence2_batch = [raw_dataset['train'][i]['sentence2'] for i in range(3)]

padded_upfront = tokenizer(sentence1_batch, sentence2_batch, padding = 'max_length', max_length = 128, truncation = True)
print(f"Original sentence1: {sentence1_batch} \n and length is: {len(padded_upfront['input_ids'][0])}\n")
print(f"Original sentence2: {sentence2_batch} \n and length is: {len(padded_upfront['input_ids'][1])}\n")

In [ ]:
raw_dataset

In [ ]:
sentence1_batch

In [ ]:
# With dynamic padding via collator
samples_new = [tokenized_datasets['train'][i] for i in range(3)]
clean_samples = [{'input_ids':s['input_ids'], 'token_type_ids':s['token_type_ids'], 'attention_mask':s['attention_mask']} for s in samples_new]
batch = data_collator(clean_samples)
print(batch['input_ids'].shape[1])

In the above example, we saw that without dynmaic padding, even though the sentences are shorter, due to the max_length = 128, extra padding tokens got added for each input until the length became 128. This results in wasted tokens per row that the model has to process and mask out for nothing.

Whereas, when used dynamic padding (collator), extra padding was added only based on the maximum length sentence from the entire input/batch hence, we saw the total token length around 59 only, which is the same length as the sentence 2. In case of sentence 1 and sentence 3 those were matched upto sentence 2 only. Sentence 2 has the maximum length among the three sentences.

In [ ]:
tokenized_datasets

In [ ]:
type(new_dict_samples)

## **TrainingArguments**

- Config object that tells `Trainer` **how** to run training: batch size, learning rate, number of epochs, when to evaluate, when to save, etc. It doesn't train anything itself; `Trainer` reads its settings from it.
- Only required argument: `output_dir` (where checkpoints and logs get saved). Everything else has a default, and `print(training_args)` shows all of them.

```python
training_args = TrainingArguments(
    output_dir='test-trainer',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-5
)
```

**Key arguments so far:**

- `num_train_epochs`: one epoch = one full pass through the entire training set. With 3668 examples and 3 epochs, the model sees the whole dataset 3 separate times.
- `per_device_train_batch_size`: how many examples are processed together in one forward/backward pass before the weights update (the average loss over those examples is used).
  - Batches per epoch = dataset size / batch size, rounded up. Here 3668 / 16 = 229.25 → **230 batches per epoch** (the last batch is smaller, with 4 examples). With 3 epochs that is 230 × 3 = **690 weight updates**.
- `learning_rate`: how big a step the model takes when updating weights after each batch (default `5e-5`).
  - Too high → overshoots, loss keeps jumping around, may never converge.
  - Too low → training is very slow and may get stuck.
  - Training from scratch → larger steps are needed (starting from random weights).
  - Fine-tuning → small steps, because pretrained weights are already good and only need nudging. Large steps risk wiping out the pretrained knowledge (catastrophic forgetting).
- `eval_strategy`: defines if and when the model should evaluate the training. By default value is `no`. The other two available options are  `steps` and `epoch`. `steps` tells every N training steps evaluate the model and `epoch` tells at the the end of each epoch and the epoch value is set in `num_train_epochs`.
- `save_strategy`: It controls when a checkpoint (a saved copy of the model's weights at that moment) is written to **output_dir**. It also takes the same options :`no`, `steps`, `epoch`.
- `load_best_model_at_end`: when set to **True** picks the single best checkpoint out of those 3 based on the scores. To pick the best one, it needs a checkpoint to exist. This is the primary reason why the values for both `eval_strategy` and `save_strategy` need to be same. If not same, assume `eval_strategy` at epoch 2 gave the best result but the checkpoint isn't saved for epoch 2 and for the next epoch 3, the checkpoint is saved but it's not optimal, we are not using the best at it's best hyperparameter values and weights resulting in not so well performing. Hence, at each evaluation, saving a checkpoints at best weights is necessary.

In [ ]:
# import TrainingArguments
from transformers import TrainingArguments

training_args_v9 = TrainingArguments(output_dir = 'test-trainer-v9', num_train_epochs = 3, per_device_train_batch_size = 16, learning_rate = 2e-5, eval_strategy = "epoch", save_strategy = "epoch",
                                  load_best_model_at_end = True)

print(training_args_v9)

## **`Trainer`**

- It is a class that actually runs the training loop. It binds together everything we have discussed above.


In [ ]:
from transformers import AutoModelForSequenceClassification
model_name_v9 = 'bert-base-uncased'
# new_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2)
new_model_v9 = AutoModelForSequenceClassification.from_pretrained(model_name_v9, num_labels = 2)
print(new_model_v9)

In [ ]:
new_model_v9

**Distinction between `bert-base-uncased` and `AutoModelFor...`**

- **`bert-base-uncased`** - The actual pretrained **model checkpoint**

- **`AutoModelFor...`** - Not a model but a **loader + head**. It:
  1. loads the pretrained body (`bert-base-uncased`'s weights)
  2. Attaches a new, **randomly-initialized head** on top. A small task-specific layer

  The body is pretrained whereas the head is brand new and gets trained during fine-tuning.

- As per the architecture, the below stays the same across every task (the body)

  - input -> 12 transformer layers, each with multiple **attention heads** (internal mechanism, unrelated to the classification head) -> final hidden state for every token.

This part is identical regardless of task - same weights, same computations whether solving MRPC (the current we are working on), QnA, NER (Named Entity Recognition - Task: Identify entities (Person, Organization, Location) in text)

- Where does the distinction happens (the head)

  - The head decides what to extract from the body's output ans how to interpret it. For example,

  | Task | What it uses from body's output | Head produces |
  |---|---|---|
  | Sequence classification (MRPC) | Only `[CLS]`'s hidden state | N class scores (`num_labels` sizes this) |
  |Token classification (NER/POS) | *Every* token's hiddens state | N tag scores, per token |
  |Question Answering | *Every* token's hidden state | 2 scores per token (start-span, end-span) |

- **The full pipeline (Sequence Clasfication example)**
  
  input text

  - -> tokenizer ->input_ids, token_type_ids, attention_mask
  - -> BERT body (12 layers, multi-head attn.) - SAME for every task
  - -> final hidden state for every token
  - -> grab [CLS]'s hidden state specifically  - task-specific split starts here
  - -> classification head (Linear -> num_labels)
  - -> 2 output scores (not_equiv/ equiv)

In short, `num_labels` doesn't tell the model *what task* to do — that's determined by which `AutoModelFor...` class you pick (which fixes the head's shape/type). `num_labels` only sizes the *output* of that head — how many scores it produces. For MRPC with `AutoModelForSequenceClassification`, we set `num_labels=2` since there are only 2 possible outcomes: `equivalent` or `not_equivalent`.

In [ ]:
tokenized_datasets['train']

Creating new dataset after the `.map()` in order to use only 'label', 'input_ids', 'token_type_ids', 'attention_mask'. For this we will use `remove_column()` method.

We also need to rename 'label' to 'labels' and for this we will use `rename_column()`.

In [21]:
new_tokenized_datasets = tokenized_datasets.remove_columns(['sentence1', 'sentence2', 'idx'])
new_tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [22]:
new_tokenized_datasets = new_tokenized_datasets.rename_column('label', 'labels')
print(new_tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})


## **Trainer**

It is a class that actually runs the training loop. It ties together everything like the model, the tokenized data, the collator and the config.

`Trainer` is a powerful high-level API provided by the transformers library, designed to simplify the process of training and fine-tuning machine learning models, particularly those based on transformer architectures. The Trainer class abstracts away the intricacies of the training loop, allowing users to focus on developing and optimizing state-of-the-art models with ease.

In [ ]:
# from transformers import Trainer

# trainer_v7 = Trainer(
#     model = new_model_v7, # passing the model that need to be used via the model loader i.e. AutoModelFor...
#     args = training_args_v7, # TrainingArguments config object - the one we defined above epochs, learning_rate, eval_strategy, save_strategy etc
#     train_dataset = new_tokenized_datasets['train'], # train split of the new_tokenized_dataset
#     eval_dataset = new_tokenized_datasets['validation'], # eval split of the new_tokenized_dataset
#     data_collator = data_collator, # handles dynamic per-batched padding
#     processing_class = tokenizer, # not used to re-tokenize (already done); kept so Trainer can save it alongside checkpoints and use it consistently for future inference
# )
# print(trainer_v7)

In [ ]:
tokenizer

In [ ]:
data_collator

## **compute_metrics**

- It is a function we write to tell the `Trainer` how to run the raw model outputs into meaningful scores (accuracy, F1 etc) during evaluation. If we don't use compute_metrics, the `Trainer` only tracks the loss.

- **How it works, conceptully** - The during evaluation, the model produces **logits**. `conpute_metrics` receives those logits along with the **true labels** and need to:

  - convert the logits into actual predicted class (in case of MRPC, pick whicher of the 2 scores is higher)
  - Compare the prediction with the true labels
  - Return a dictionary of metric names to values.

- The `evaluate` library (not part of `transformers`) provides ready made metric function so we don't have to implement accuracy/F1 from scratch.

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


### **`evaluate` library**

- (separate from transformers) provides ready-made metric functions. It has ready-made metric functions for the GLUE task (such as for our MRPC). It helps us from writing the metrics manually. In case of non-GLUE tasks or completely new task, we can use standalone metrics like `evaluate.load("accuracy")`, `evaluate.load("f1")`, `evaluate.load("rouge")`, `evaluate.load("bleu")`, `evaluate.load("precision")`, `evaluate.load("recall")`, etc.
- For GLUE tasks, it already knows which tasks uses which metrics.

In [4]:
import evaluate

metric = evaluate.load("glue", "mrpc")
print(metric)

EvaluationModule(name: "glue", module_type: "metric", features: {'predictions': Value('int64'), 'references': Value('int64')}, usage: """
Compute GLUE evaluation metric associated to each GLUE dataset.
Args:
    predictions: list of predictions to score.
        Each translation should be tokenized into a list of tokens.
    references: list of lists of references for each translation.
        Each reference should be tokenized into a list of tokens.
Returns: depending on the GLUE subset, one or several of:
    "accuracy": Accuracy
    "f1": F1 score
    "pearson": Pearson Correlation
    "spearmanr": Spearman Correlation
    "matthews_correlation": Matthew Correlation
Examples:

    >>> glue_metric = evaluate.load('glue', 'sst2')  # 'sst2' or any of ["mnli", "mnli_mismatched", "mnli_matched", "qnli", "rte", "wnli", "hans"]
    >>> references = [0, 1]
    >>> predictions = [0, 1]
    >>> results = glue_metric.compute(predictions=predictions, references=references)
    >>> print(results

- **`compute_metrics`**: it's a function we write, but the `Trainer` calls it automatically during every evaluation (per `eval_strategy`).
  - `Trainer` passes it one argument, an `EvalPrediction` (`eval_preds`) object with:
    - `.predictions` -> the raw logits for the whole eval set, shape `(num_examples, num_labels)` - example, `(408, 2)` for MPRC **validation**
    - `.label_ids` -> true labels, shape (`num_examples,)`
  - **Logits** are **raw scores per class** not probabilities - column 0 = score for class 0, column 1 = score for class 1. (matches `classifier`'s `out_features = num_labels`)
  - Convert logits to predicted class with `np.argmax(logits, axis=-1)`. `axis=-1` picks the max **within each row** (i.e. within one example's class scores)- not across examples.
  - Function return whatever `metric.compute()` gives back a **dictionary**, e.g. `{'acuuracy':8.85, 'f1':0.90}`. `Trainer` uses this for logging and (if `metric_forbest_model` is set) for picking the best checkpoint.

In [5]:
import numpy as np

def compute_metrics(eval_preds):

  logits, labels = eval_preds.predictions, eval_preds.label_ids
  predictions = np.argmax(logits, axis = -1)
  # print(logits.shape)
  return metric.compute(predictions = predictions, references = labels)

In [ ]:
results_v7 = compute_metrics(trainer_v9.predict(new_tokenized_datasets['validation']))
print(results_v7)

- The `compute_metrics` must be passed in the `Trainer` (either at construction as show below or set afterward via 'trainer.compute_metrics = compute_metrics`)

In [ ]:
from transformers import Trainer

trainer_v9 = Trainer(
    model = new_model_v9, # passing the model that need to be used via the model loader i.e. AutoModelFor...
    args = training_args_v9, # TrainingArguments config object - the one we defined above epochs, learning_rate, eval_strategy, save_strategy etc
    train_dataset = new_tokenized_datasets['train'], # train split of the new_tokenized_dataset
    eval_dataset = new_tokenized_datasets['validation'], # eval split of the new_tokenized_dataset
    data_collator = data_collator, # handles dynamic per-batched padding
    processing_class = tokenizer, # not used to re-tokenize (already done); kept so Trainer can save it alongside checkpoints and use it consistently for future inference
    compute_metrics = compute_metrics, # passing the compute_metrics function
)
print(trainer_v9)

In [ ]:
trainer_v9.train()

In [ ]:
print(f"The optimal checkpoint was: {trainer_v9.state.best_model_checkpoint} (step 230 = end of epoch 1) \n")
print(f"Best validation loss: {trainer_v9.state.best_metric}")
print(f"Total epochs trained: {trainer_v9.state.epoch}")  # this is just training's final progress, not tied to the best checkpoint

- Validation loss **increased** each epoch → sign of overfitting as training continued; accuracy/F1 stayed roughly flat, no real improvement past epoch 1.
- Since `metric_for_best_model` defaults to `eval_loss` (lower = better), `load_best_model_at_end=True` loaded **epoch 1's checkpoint** (`checkpoint-230`, loss 0.350) — not the final epoch — confirming exactly why this setting matters: "trained longer" ≠ "better."
- `trainer.state.epoch` reports training's overall progress (3.0 = fully trained), not which epoch produced the best checkpoint — that's found via `trainer.state.best_model_checkpoint` instead.

In [ ]:
metric.compute

In [ ]:
import torch
import numpy as np

a = torch.randint(2, 6, (2, 5))
print(a)
print(np.argmax(a, axis=0), np.argmax(a, axis=1), np.argmax(a, axis=-1))

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# push to HF hub
# repo_name = "bert-base-uncased-finetuned-mrpc"
# trainer.model.push_to_hub(repo_name)
# tokenizer.push_to_hub(repo_name)

### Importing our model and testing it's working!

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_name = "GoingDeep/bert-base-uncased-finetuned-mrpc"

tokenizer = AutoTokenizer.from_pretrained(repo_name)

model = AutoModelForSequenceClassification.from_pretrained(repo_name)

In [ ]:
# testing on new pairs of sentences

sentence1 = "This cup of coffee was quite tasty!"
sentence2 = "The coffee in this cup tasted pretty good!"

# # Should be: equivalent
# sentence1 = "The company reported a sharp rise in quarterly profits."
# sentence2 = "Quarterly profits at the company rose sharply, it reported."

# Should be: not_equivalent
# sentence1 = "The company reported a sharp rise in quarterly profits."
# sentence2 = "The city council approved a new budget for road repairs."

In [ ]:
inputs = tokenizer(sentence1, sentence2, return_tensors = "pt")

id2label = {0: "not_equivalent", 1: "equivalent"}

with torch.inference_mode():
  outputs = model(**inputs)
# print(outputs)
  pred_ids = torch.argmax(outputs.logits, axis=-1)
  label = id2label[pred_ids.item()]
print(f"prediction: {label}\n")

In [ ]:
print(trainer_v9.args.learning_rate, trainer_v9.args.per_device_train_batch_size, trainer_v9.args.num_train_epochs)

In [44]:
def to_instruction(row):
    lookup = {0: "The sentences are not paraphrased", 1: "The sentences are paraphrased"}
    response_text = lookup[row["label"]]   # label -> response string
    prompt = f"Please tell if sentence1: {row["sentence1"]} and sentence2: {row["sentence2"]} are paraphrased or not"                                   # your template, with {row['sentence1']} and {row['sentence2']}
    return {"prompt": prompt, "response": response_text}

In [45]:
new_result_1 = to_instruction(raw_dataset['train'][0])
print(new_result_1)

{'prompt': 'Please tell if sentence1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence . and sentence2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence . are paraphrased or not', 'response': 'The sentences are paraphrased'}


In [37]:
print(to_instruction(raw_dataset['train'][1]))
print(raw_dataset['train'][1]['sentence1'])

{'prompt': "Please tell if Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .: and Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .: are paraphrased or not", 'response': 'The sentences are not paraphrased'}
Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .


In [52]:
instruction_dataset = raw_dataset['train'].map(to_instruction)
print(instruction_dataset)

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx', 'prompt', 'response'],
    num_rows: 3668
})


In [53]:
instruction_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0,
 'prompt': 'Please tell if sentence1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence . and sentence2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence . are paraphrased or not',
 'response': 'The sentences are paraphrased'}

In [54]:
new_instruction_dataset = raw_dataset['train'].map(to_instruction, remove_columns=['sentence1', 'sentence2', 'label', 'idx'])
print(new_instruction_dataset)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'response'],
    num_rows: 3668
})


In [55]:
new_instruction_dataset[0]

{'prompt': 'Please tell if sentence1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence . and sentence2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence . are paraphrased or not',
 'response': 'The sentences are paraphrased'}

## Instruction Fine-Tuning Format (prompt/response pairs)

- **Goal:** teach a model to follow instructions by training on many examples of "instruction in, answer out".
- **Format:** each example is two pieces of text, `{'prompt': '...', 'response': '...'}`.
  - prompt = what the user asks
  - response = what the model should write back
- **Training idea:** the model reads the prompt followed by the response as one text and learns to predict the response tokens, one next token at a time. After training, you give it only a prompt and it writes the response itself.
- **Compared with my MRPC classifier:**
  - Classifier: BERT body + classification head, output is a fixed class (0 or 1), and I translate it to a word with `id2label`.
  - Instruction fine-tuning: a decoder model (`AutoModelForCausalLM`, like Llama from Week 6), output is free text, no `num_labels`. The two are separate models and can't be chained.
- **Building the data from MRPC:**
```python
  def to_instruction(row):
      response_text = {0: "The sentences are not paraphrased", 1: "The sentences are paraphrased"}[row["label"]]
      prompt = f"Please tell if sentence1: {row['sentence1']} and sentence2: {row['sentence2']} are paraphrased or not"
      return {"prompt": prompt, "response": response_text}

  new_instruction_dataset = raw_dataset['train'].map(to_instruction, remove_columns=['sentence1', 'sentence2', 'label', 'idx'])
```
  - `{0: ..., 1: ...}[row["label"]]` builds a dict and looks up one value in the same line. It turns the label number into response text.
  - The function reads from `row`, not from a fixed row like `raw_dataset['train'][0]`. Hardcoding row 0 would give every prompt the same sentences.
  - `.map()` calls the function once per row and adds the returned keys as new columns. `remove_columns` drops the old ones afterwards.
- **The function doesn't judge anything.** The paraphrase answer comes from the human-written `label` column. The function only reformats it as text. The model learns to judge during training, from thousands of these pairs.
- **Prompt wording:** keep the template identical for every row, so only the two sentences change. Putting each sentence on its own labeled line (`\nsentence1: ...`) makes the boundary between them clearer.
- **Not done here:** no model was loaded or trained. The next real steps would be to join prompt and response into one text, tokenize it, and train a causal LM on it.

In [6]:
from transformers import AutoModelForSequenceClassification

In [25]:
from transformers import DataCollatorWithPadding
from transformers import Trainer
from transformers import set_seed

# import TrainingArguments
from transformers import TrainingArguments

training_args_baseline = TrainingArguments(output_dir = 'test-trainer-baseline')

set_seed(42)   # before loading, so the random head is reproducible
model_name_fresh = 'bert-base-uncased'
fresh_model = AutoModelForSequenceClassification.from_pretrained(model_name_fresh, num_labels=2)
baseline_trainer = Trainer(model=fresh_model, args=training_args_baseline, data_collator=data_collator, compute_metrics=compute_metrics)
baseline_results = baseline_trainer.evaluate(eval_dataset=
new_tokenized_datasets['validation'])   # returns eval_accuracy and eval_f1
print(baseline_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.634252,0,0.683824,0.812227


{'eval_loss': 0.6342515349388123, 'eval_accuracy': 0.6838235294117647, 'eval_f1': 0.8122270742358079}
